# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
# ! pip install -q schedule pytest # установка библиотек, если ещё не
! pip3 install schedule pytest

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin

/Users/petrova.kseniya16/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [24]:
 def get_book_data(book_url: str) -> dict:
    """
    Собирает информацию о книге со страницы товара на сайте Books to Scrape.

    Выполняется запрос по переданному URL, проверяется успешность ответа.
    С помощью BeautifulSoup и извлекает основные сведения
    о книге, а также все характеристики из таблицы Product Information. 
    Если какие-то данные отстутвуют они заменяются дефолтными значениями

    Args:
        book_url (str): Прямая ссылка на страницу конкретной книги.

    Returns:
        dict: Словарь с данными о книге, включающий ключи:
            - 'title' (str): Название книги.
            - 'price' (str): Цена книги с символом валюты.
            - 'rating' (int): Рейтинг книги (0 - если не найден).
            - 'availability' (str): Информация о наличии и количестве экземпляров.
            - 'description' (str): Текстовое описание книги.
            - 'product_information' (dict): Словарь характеристик из таблицы
              Product Information (UPC, Product Type, Price, Tax, Availability,
              Number of reviews).
            Если запрос не удался, возвращается пустой словарь.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    try:
        response = requests.get(book_url, timeout=20)
    except requests.RequestException as e:
        print(f"Не удалось загрузить книгу {book_url}: {e}")
        return {}
    
    # Проверка статуса запроса
    if response.status_code != 200:
        print(f"Request error: {response.status_code}")
        return {}

    soup = BeautifulSoup(response.content, "html.parser")

    # Название книги
    product_main = soup.find("div", class_="product_main")
    title_tag = product_main.find("h1") if product_main else None
    title = title_tag.get_text(strip=True) if title_tag else ""

    # Цена
    price_tag = soup.find("p", class_="price_color")
    price = price_tag.get_text(strip=True) if price_tag else ""

    # Рейтинг (хранится во втором классе тега <p class="star-rating Three">)
    rating_tag = soup.find("p", class_="star-rating")
    rating_classes = rating_tag.get("class", []) if rating_tag else []
    rating_name = rating_classes[1] if len(rating_classes) > 1 else "Unknown"
    rating_dict = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }
    rating = rating_dict.get(rating_name, 0)

    # Наличие (количество в наличии)
    availability_tag = soup.find("p", class_="instock availability")
    availability = availability_tag.get_text(strip=True) if availability_tag else ""

    # Описание книги
    description_header = soup.find("div", id="product_description")
    if description_header:
        description_tag = description_header.find_next_sibling("p")
        description = description_tag.get_text(strip=True) if description_tag else ""
    else:
        description = ""

    # Дополнительные характеристики из таблицы Product Information
    product_info = {}
    info_table = soup.find("table", class_="table table-striped")
    if info_table:
        for row in info_table.find_all("tr"):
            header_cell = row.find("th")
            value_cell = row.find("td")
            if header_cell and value_cell:
                key = header_cell.get_text(strip=True)
                value = value_cell.get_text(strip=True)
                product_info[key] = value

    return {
        "title": title,
        "price": price,
        "rating": rating,
        "availability": availability,
        "description": description,
        "product_information": product_info,
    }
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 3,
 'availability': 'In stock (22 available)',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place 

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [25]:
def scrape_books(is_save: bool = False):
    """
    Сбор информацию обо всех книгах из каталога Books to Scrape.

    Проходим по всем страницам каталога и извлекаем ссылки на книг.
    С помощью функции get_book_data извлекаем данные о книги. 
    При необходимости сохраняет результат в файл.

    Args:
        is_save (bool): Флаг сохранения результата. Если True данные записываются в файл 'books_data.txt' в формате JSON.
        По умолчанию значение False.

    Returns:
        list: Список с информацией о каждой книге.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    base_url = "http://books.toscrape.com/catalogue/"
    books = []
    page_number = 1

    while True:
        # URL текущей страницы каталога
        page_url = f"{base_url}page-{page_number}.html"

        try:
            response = requests.get(page_url, timeout=10)
        except requests.RequestException as e:
            print(f"Ошибка соединения на странице {page_number}: {e}")
            break

        # Если страницы не существует (конец каталога), штатно выходим
        if response.status_code == 404:
            print("Каталог полностью пройден.")
            break

        if response.status_code != 200:
            print(f"Request error: {response.status_code}")
            break

        soup = BeautifulSoup(response.content, "html.parser")
        # Теги ссылок на страницы всех книг на текущей странице
        book_links = soup.select("article.product_pod h3 a")

        # Проходим по каждой ссылке и собираем данные о книге
        for link in book_links:
            href = link.get("href")
            if not href:
                continue

            book_url = urljoin(base_url, href)
            book_data = get_book_data(book_url)

            if book_data:
                books.append(book_data)

        print(f"Обработана страница {page_number}")

        # next_page = soup.find("li", class_="next")
        # if not next_page:
        #     break

        page_number += 1

     # Сохраняем собранный список книг в текстовый файл, если необходимо
    if is_save:
        with open("books_data.txt", "w", encoding="utf-8") as file:
            json.dump(books, file, ensure_ascii=False, indent=4)

    return books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [26]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Обработана страница 1
Обработана страница 2
Обработана страница 3
Обработана страница 4
Обработана страница 5
Обработана страница 6
Обработана страница 7
Обработана страница 8
Обработана страница 9
Обработана страница 10
Обработана страница 11
Обработана страница 12
Обработана страница 13
Обработана страница 14
Обработана страница 15
Обработана страница 16
Обработана страница 17
Обработана страница 18
Обработана страница 19
Обработана страница 20
Обработана страница 21
Обработана страница 22
Обработана страница 23
Не удалось загрузить книгу http://books.toscrape.com/catalogue/millers-valley_534/index.html: HTTPConnectionPool(host='books.toscrape.com', port=80): Max retries exceeded with url: /catalogue/millers-valley_534/index.html (Caused by ConnectTimeoutError(<HTTPConnection(host='books.toscrape.com', port=80) at 0x1135d5040>, 'Connection to books.toscrape.com timed out. (connect timeout=20)'))
Обработана страница 24
Обработана страница 25
Обработана страница 26
Обработана страница 

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [27]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def run_scheduler(run_time: str = "19:00") -> None:
    """
    Запускает планировщик для ежедневного автоматического сбора данных о книгах.

    Args:
        run_time (str): Время ежедневного запуска в формате 'ЧЧ:ММ'.
            По умолчанию '19:00'.
    """

    # Очищаем предыдущие задачи планировщика
    schedule.clear()

    # Настраиваем ежедневный запуск парсера с сохранением данных
    schedule.every().day.at(run_time).do(scrape_books, is_save=True)

    print(f"Планировщик запущен. Сбор данных запланирован на {run_time}.")

    try:
        while True:
            schedule.run_pending()
            time.sleep(30)

    except KeyboardInterrupt:
        print("Работа планировщика остановлена вручную.")


# Запуск планировщика на 19:00
run_scheduler()
# run_scheduler("17:56") использовалось для локальной проверки
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Планировщик запущен. Сбор данных запланирован на 19:00.
Работа планировщика остановлена вручную.


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [31]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
#! pytest test/test_scraper.py
! pytest tests/test_scraper.py # по задание надо в папке tests
# ! python3 -m pytest -v tests/test_scraper.py

# python3 -m pytest



============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0
rootdir: /Users/petrova.kseniya16/HW_03_python_ds
plugins: anyio-4.12.1
collected 4 items                                                              

tests/test_scraper.py ....                                               [100%]

=============================== warnings summary ===============================
../Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35
  /Users/petrova.kseniya16/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 4 passed, 1 warning in 0.12s =========================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Надо было написать парсер для сайта с книгами, настроить выгрузку в 19 и покрыть тестами. Также надо было выложить проект на гитхаб. Этап с тестами вызывал некоторые сложности так как вообще не было опыта. Гитхаб тоже неожилано вызвал сложности, но всё прошло успешно .`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Сначала застряла сразу, с запуска загрузки библиотеки у меня надо написать pip3. Оставила изначальный вариает в комментарии. При выполнении первого задания, решила, что рейтинг надо числом, на всякий случай пробежалась по всем вариантам звёзд, чтобы составить словарь. Также, чтобы прогромма не падала, если вдруг на сайте забыли добавить какие-то данные, решила действовать поэтомпно сначала найти данные, если есть только потом обратиться. Я восприняла задание, как мой пет проект, где я делаю библиотеку и хочу максимальный ассортиимент и для меня не страшно, если какие-то данные будут неполные. Но в некоторых случаях возможно не полные данные логичнее просто пропустить. При выполнении второго задания, сначала сделала обработку ошибки по аналогии с первой, а потом поняла, что если при загрузки страницы интернет моргнёт, то потеряется вся выгрузка, поэтому переписала обработку. Также выгружать все страницы оказалось долго, поэтому при проверке результата парсила только первую страницу. Также при выполнении задания 2, нашла soup.find("li", class_="next"), который говорит о том если ли кнопка переключеничения на след струницу. Но при разработке решила, что это повышает риск, так как дизайн могут изменить, парсинг в принципе не очень надежный в этом плане. Поэтому решила перестраховаться и сделать проверку на 404, в таком случае получилась бы двойная проверка, поэтому с class_="next" решила закоментировать. Когда во втором задани, решила сделать проверку и выгрузить всё, некоторые книги падали с ошибкой ConnectTimeoutError. Я установила timeout=20 и добавила обработку исключения, которую забыла добавить раньше. Можно было бы убрать timeout, но мне кажется это не очень хороший патерн). В Задании 3 не сказано, что надо написать функцию, но я решила, что так будет удобнее для дебага, да и в принципе для использования. В этом дз прослеживается больше возможности делать на своё усмотрение, поэтому я думаю это не снизит оценку). При локальном запуске несколько раз, вспомнила, что важно отчищать расписоние, поэтому добавила schedule.clear(). Задание с тестами было сложное, сначала не запускалось, потом ошибки, потом гонялось очень долго. Изначально я выяснила, что совершила ошибку при написании тестов, я возвращала всегда 200, а конец каталога у меня был на проверку на 404. Так как проверку soup.find("li", class_="next") я закоментировала, получался вечный цикл. Сдать задание в гите оказалось не так просто. Я сейчас каждый день с ним работаю, но я очень давно не создала в репозиторий сама. Обычно если раньше надо было сдать дз в гитлабе, я просто загружала туда готовый проект полностью. Когда я начала делать это задание, я изначально думала сделать также, а потом оказалось что всё не так просто... когда я запущила проект на ветку, оказалось что это единственная ветка и main нет... Спросила у ии что делать, запушила main.. оказалось что ветки не сравнить потому что у них нет ничего общего.... Того что должен быть общий коммит я не знала.. снова к ии.. и по итогу получилось... Надо было запустить все ячейки, я по привычке сделала рестарт и вся выгрузка началась заново... сглупила да, она выгружалась 11 минут... 
Долго ждала чтобы отправить, при этом в гитхабе не отображается нормально, там будто бы какие-то ошибки... при этом когда я перезапускала какие-то ячейки и они меняли свой вывод, изменеия в дифе не отображались.... 
Оказалось когда я формировала файлы как надо по структуре, у меня задвоились файлы я отправила в гит не самый акутальный файл(( Очень печально`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: При написании тестов обращалась к лекции. Возникла проблема с ModuleNotFoundError: No module named 'scraper', спросила у ии как решить, подсказал решение sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))). Также просила подсказать команды гита, чтобы запушить данные в гитхаб. "Я запушила ветку в гитхаб, однако при попытки созадния mr возникает такая напдпись main and hw-books-parser are entirely different commit histories."`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Есть сложности с написание тестов, не могу точно сказать в чём именно проблема, думаю не хватает практики.
`

**5. Самооценка по критериям задания**:

`Ответ: В прицнипе всё получилось, были проблемы с тестами, пофиксила нескаолько багов в процессе разработки, для некотрых решений представила альтернативный вариант. Были проблемы с тестами, неправильно написала условия, забыла что условия выхода другие. Но исправилась же... Гитхаб тоже по итогу смогла победить. Оцениваю на 10 баллов

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Неприятно была долгая выгрузка данных, непонятно ошибка в коде или данные просто большие`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

